# Logistic Regression Prediction Notebook
Bu not defteri, kaydedilmiş lojistik regresyon modelleriyle PlantVillage ve PlantDoc görüntüleri üzerinde tahmin yapar. Aşağıdaki hücreleri sırayla çalıştırarak modelleri yükleyip test görsellerini deneyebilirsin.


## 🧩 Cell 1 – Import'lar


In [72]:
import numpy as np
from PIL import Image
import os
import math
import random


## 🧩 Cell 2 – Logistic Regression sınıfı (load ile uyumlu)
Not: Buradaki `load` metodu, training tarafında `save()` ile kaydettiğin `.npz` dosyasıyla uyumlu.


In [73]:
def softmax(logits):
    max_logit = max(logits)
    exps = [math.exp(z - max_logit) for z in logits]
    s = sum(exps)
    if s == 0.0:
        c = len(logits)
        return [1.0 / c for _ in range(c)]
    return [e / s for e in exps]


class MulticlassLogisticRegression:
    def __init__(self, num_features, num_classes, learning_rate=0.1):
        self.num_features = num_features
        self.num_classes = num_classes
        self.learning_rate = learning_rate
        # dummy init, real weights will be loaded from file
        self.W = [
            [(random.random() - 0.5) * 0.01 for _ in range(num_classes)]
            for _ in range(num_features)
        ]
        self.b = [(random.random() - 0.5) * 0.01 for _ in range(num_classes)]

    def _compute_logits(self, x):
        logits = [0.0 for _ in range(self.num_classes)]
        for k in range(self.num_classes):
            s = 0.0
            for j in range(self.num_features):
                s += x[j] * self.W[j][k]
            s += self.b[k]
            logits[k] = s
        return logits

    def predict_proba_one(self, x):
        logits = self._compute_logits(x)
        probs = softmax(logits)
        return probs

    def predict_one(self, x):
        probs = self.predict_proba_one(x)
        best_class = 0
        best_prob = probs[0]
        for k in range(1, self.num_classes):
            if probs[k] > best_prob:
                best_prob = probs[k]
                best_class = k
        return best_class

    def predict(self, X):
        return [self.predict_one(x) for x in X]

    @classmethod
    def load(cls, path):
        data = np.load(path)
        num_features = int(data["num_features"])
        num_classes = int(data["num_classes"])
        model = cls(
            num_features=num_features,
            num_classes=num_classes,
            learning_rate=0.01
        )
        W_array = data["W"]
        b_array = data["b"]
        model.W = W_array.tolist()
        model.b = b_array.tolist()
        print(f"Model loaded from {path}")
        print("num_features:", num_features)
        print("num_classes:", num_classes)
        return model


## 🧩 Cell 3 – Görüntüyü 32×32 gri vektöre çevirme


In [74]:
IMG_SIZE = 32

def load_image_as_vector(path, img_size=IMG_SIZE):
    with Image.open(path) as img:
        img = img.convert("L")
        img = img.resize((img_size, img_size))
        pixels = list(img.getdata())
        vector = [p / 255.0 for p in pixels]
        return vector


## 🧩 Cell 4 – PlantVillage sınıf isimlerini klasörden oku
Bu, PlantVillage için tüm sınıfları (klasör isimlerini) listeleyecek ✅


In [75]:
def get_class_names(root_dir):
    names = []
    for item in os.listdir(root_dir):
        full_path = os.path.join(root_dir, item)
        if os.path.isdir(full_path):
            names.append(item)
    names = sorted(names)
    return names

# predict.ipynb, "Source Code" klasöründe ise:
PLANTVILLAGE_ROOT = "../Dataset/plantvillage/color"
class_names_pv = get_class_names(PLANTVILLAGE_ROOT)
idx_to_class_pv = {idx: name for idx, name in enumerate(class_names_pv)}

print("PlantVillage classes:")
for idx, name in enumerate(class_names_pv):
    print(f"{idx:2d} -> {name}")


PlantVillage classes:
 0 -> Apple___Apple_scab
 1 -> Apple___Black_rot
 2 -> Apple___Cedar_apple_rust
 3 -> Apple___healthy
 4 -> Blueberry___healthy
 5 -> Cherry_(including_sour)___Powdery_mildew
 6 -> Cherry_(including_sour)___healthy
 7 -> Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
 8 -> Corn_(maize)___Common_rust_
 9 -> Corn_(maize)___Northern_Leaf_Blight
10 -> Corn_(maize)___healthy
11 -> Grape___Black_rot
12 -> Grape___Esca_(Black_Measles)
13 -> Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
14 -> Grape___healthy
15 -> Orange___Haunglongbing_(Citrus_greening)
16 -> Peach___Bacterial_spot
17 -> Peach___healthy
18 -> Pepper,_bell___Bacterial_spot
19 -> Pepper,_bell___healthy
20 -> Potato___Early_blight
21 -> Potato___Late_blight
22 -> Potato___healthy
23 -> Raspberry___healthy
24 -> Soybean___healthy
25 -> Squash___Powdery_mildew
26 -> Strawberry___Leaf_scorch
27 -> Strawberry___healthy
28 -> Tomato___Bacterial_spot
29 -> Tomato___Early_blight
30 -> Tomato___Late_blight
31 -> T

## 🧩 Cell 5 – PlantDoc için sınıf listesi
Burada iki seçenek var:

- Henüz net isim listemiz yoksa → şimdilik generic isim üretelim.
- İleride istersen `plantdoc_classes.txt` gibi bir dosyada gerçek isimleri tutup buradan okuyabiliriz.

Şimdilik generic (ama tüm sınıfları yazdıran) versiyon. Bu kısmı bir sonraki hücrede tamamlayacağız; önce modelleri yükleyelim.


## 🧩 Cell 6 – Modelleri dosyadan yükle
Bu hücreyi mutlaka çalıştır. Aksi halde `NameError: model_pv is not defined` alırsın.


In [76]:
# Models are saved under "models" folder next to this notebook
MODEL_PV_PATH = "models/logreg_plantvillage.npz"
MODEL_PD_PATH = "models/logreg_plantdoc.npz"

model_pv = MulticlassLogisticRegression.load(MODEL_PV_PATH)
model_pd = MulticlassLogisticRegression.load(MODEL_PD_PATH)


Model loaded from models/logreg_plantvillage.npz
num_features: 1024
num_classes: 38
Model loaded from models/logreg_plantdoc.npz
num_features: 1024
num_classes: 28


## 🧩 Cell 7 – PlantDoc sınıf isimlerini üret ve yazdır
Model yüklendikten sonra `model_pd.num_classes` üzerinden sınıf sayısını biliyoruz.
Şimdilik isimleri `PlantDoc_class_0, PlantDoc_class_1, …` gibi yapalım. İleride gerçek isimleri `plantdoc_classes.txt` dosyasından okuyabiliriz.


In [77]:
# Path to PlantDoc train labels
PLANTDOC_ROOT = "../Dataset/plantdoc"
train_labels_dir = os.path.join(PLANTDOC_ROOT, "train", "labels")

orig_ids_set = set()

for fname in os.listdir(train_labels_dir):
    if not fname.lower().endswith(".txt"):
        continue
    label_path = os.path.join(train_labels_dir, fname)
    try:
        with open(label_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                class_id = int(parts[0])
                orig_ids_set.add(class_id)
    except FileNotFoundError:
        continue

unique_orig_ids = sorted(orig_ids_set)

print("Original class ids seen in PlantDoc TRAIN split:", unique_orig_ids)
print("Count (should match model_pd.num_classes):", len(unique_orig_ids))
print("model_pd.num_classes:", model_pd.num_classes)


Original class ids seen in PlantDoc TRAIN split: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
Count (should match model_pd.num_classes): 30
model_pd.num_classes: 28


In [78]:
plantdoc_class_names = [
    "Apple Scab Leaf",                # 0
    "Apple leaf",                     # 1
    "Apple rust leaf",                # 2
    "Bell_pepper leaf spot",          # 3
    "Bell_pepper leaf",               # 4
    "Blueberry leaf",                 # 5
    "Cherry leaf",                    # 6
    "Corn Gray leaf spot",            # 7
    "Corn leaf blight",               # 8
    "Corn rust leaf",                 # 9
    "Peach leaf",                     # 10
    "Potato leaf early blight",       # 11
    "Potato leaf late blight",        # 12
    "Potato leaf",                    # 13
    "Raspberry leaf",                 # 14
    "Soyabean leaf",                  # 15
    "Soybean leaf",                   # 16
    "Squash Powdery mildew leaf",     # 17
    "Strawberry leaf",                # 18
    "Tomato Early blight leaf",       # 19
    "Tomato Septoria leaf spot",      # 20
    "Tomato leaf bacterial spot",     # 21
    "Tomato leaf late blight",        # 22
    "Tomato leaf mosaic virus",       # 23
    "Tomato leaf yellow virus",       # 24
    "Tomato leaf",                    # 25
    "Tomato mold leaf",               # 26
    "Tomato two spotted spider mites leaf",  # 27
    "grape leaf black rot",           # 28
    "grape leaf"                      # 29
]

print("Length of plantdoc_class_names:", len(plantdoc_class_names))


Length of plantdoc_class_names: 30


In [79]:
# new_index (0..num_classes-1) -> original_id (0..29)
idx_to_orig_id_pd = {new_idx: orig_id for new_idx, orig_id in enumerate(unique_orig_ids)}

# new_index -> human-readable class name from data.yaml
idx_to_class_pd = {
    new_idx: plantdoc_class_names[orig_id]
    for new_idx, orig_id in idx_to_orig_id_pd.items()
}

print("\nPlantDoc model index -> (orig_id, class_name):")
for new_idx in range(model_pd.num_classes):
    orig_id = idx_to_orig_id_pd[new_idx]
    name = idx_to_class_pd[new_idx]
    print(f"{new_idx:2d} -> {orig_id:2d} -> {name}")



PlantDoc model index -> (orig_id, class_name):
 0 ->  0 -> Apple Scab Leaf
 1 ->  1 -> Apple leaf
 2 ->  2 -> Apple rust leaf
 3 ->  3 -> Bell_pepper leaf spot
 4 ->  4 -> Bell_pepper leaf
 5 ->  5 -> Blueberry leaf
 6 ->  6 -> Cherry leaf
 7 ->  7 -> Corn Gray leaf spot
 8 ->  8 -> Corn leaf blight
 9 ->  9 -> Corn rust leaf
10 -> 10 -> Peach leaf
11 -> 11 -> Potato leaf early blight
12 -> 12 -> Potato leaf late blight
13 -> 13 -> Potato leaf
14 -> 14 -> Raspberry leaf
15 -> 15 -> Soyabean leaf
16 -> 16 -> Soybean leaf
17 -> 17 -> Squash Powdery mildew leaf
18 -> 18 -> Strawberry leaf
19 -> 19 -> Tomato Early blight leaf
20 -> 20 -> Tomato Septoria leaf spot
21 -> 21 -> Tomato leaf bacterial spot
22 -> 22 -> Tomato leaf late blight
23 -> 23 -> Tomato leaf mosaic virus
24 -> 24 -> Tomato leaf yellow virus
25 -> 25 -> Tomato leaf
26 -> 26 -> Tomato mold leaf
27 -> 27 -> Tomato two spotted spider mites leaf


## 🧩 Cell 8 – Tahmin fonksiyonları (iki model için)


In [80]:
def predict_with_plantvillage(image_path):
    """
    Predict class for a given image using PlantVillage model (model_pv).
    Returns predicted index and class name.
    """
    x_vec = load_image_as_vector(image_path, img_size=IMG_SIZE)
    pred_idx = model_pv.predict_one(x_vec)
    class_name = idx_to_class_pv.get(pred_idx, f"unknown_{pred_idx}")
    print(f"[PlantVillage] predicted class index = {pred_idx}")
    print(f"[PlantVillage] predicted class name  = {class_name}")
    return pred_idx, class_name


def predict_with_plantdoc(image_path):
    """
    Predict class for a given image using PlantDoc model (model_pd).
    Uses correct mapping from model index -> original id -> yaml class name.
    """
    x_vec = load_image_as_vector(image_path, img_size=IMG_SIZE)
    pred_idx = model_pd.predict_one(x_vec)

    orig_id = idx_to_orig_id_pd.get(pred_idx, None)
    if orig_id is None:
        class_name = f"unknown_index_{pred_idx}"
    else:
        class_name = plantdoc_class_names[orig_id]

    print(f"[PlantDoc] predicted model index = {pred_idx}")
    print(f"[PlantDoc] original class id     = {orig_id}")
    print(f"[PlantDoc] predicted class name  = {class_name}")
    return pred_idx, class_name



## 🧩 Cell 9 – Gerçek test
Hem PlantVillage hem PlantDoc modeli dosyadan yüklenecek, tüm sınıflar bir kez konsola yazılacak, seçtiğin test görseli için iki modelin tahmini de görünecek.


In [81]:
test_image_path = "../TestImages/test3.jpg"  # kendi path'ini gir

print("Testing with PlantVillage model:")
pv_idx, pv_name = predict_with_plantvillage(test_image_path)

print("\nTesting with PlantDoc model:")
pd_idx, pd_name = predict_with_plantdoc(test_image_path)


Testing with PlantVillage model:
[PlantVillage] predicted class index = 26
[PlantVillage] predicted class name  = Strawberry___Leaf_scorch

Testing with PlantDoc model:
[PlantDoc] predicted model index = 10
[PlantDoc] original class id     = 10
[PlantDoc] predicted class name  = Peach leaf
